## # Requirement 3: Spatial Attribution of Cleaned GPS Tracks

This section performs the spatial attribution of the cleaned GPS tracks to the administrative and settlement reference layers. After completing the quality assurance (QA) process in Requirement 2, the resulting dataset (`gps_tracks_clean`) represents a reliable set of movement points that can be used to determine which settlements were visited during the campaign.

Spatial attribution is essential because GPS points on their own do not contain information about the administrative units or settlements they fall within. By overlaying the cleaned GPS tracks with the settlement, ward, LGA, and state boundary layers, each point can be assigned to its corresponding geographic unit. This enables downstream analysis such as identifying visited settlements, summarising coverage, and supporting operational decision‑making.

## Objectives of Spatial Attribution

1. **Assign each cleaned GPS point to a settlement**  
   Using a spatial join, each point is matched to the settlement polygon it falls within. This allows identification of visited settlements and supports coverage analysis.

2. **Assign each point to its administrative units**  
   Points are attributed to ward, LGA, and state boundaries to ensure hierarchical consistency and enable aggregation at multiple administrative levels.

3. **Identify visited settlements**  
   By grouping attributed points by settlement ID, we determine which settlements were visited, how many GPS fixes were recorded in each, and the time range of visits.

4. **Handle near‑miss points using buffer tolerance**  
   Due to GPS positional error (typically 5–30 m), some points may fall just outside settlement polygons. A defensible buffer tolerance is applied to ensure genuine visits are not excluded due to minor GPS drift.

The following Python libraries are used in this section:

- **pandas** — for tabular operations, grouping, and summarising visited settlements  
- **geopandas** — for spatial joins, CRS handling, and reading/writing GeoPackage layers  
- **shapely** — for geometric operations such as buffering and spatial predicates  
- **fiona** — for listing and verifying layers within the GeoPackage  
- **matplotlib / seaborn** (optional) — for diagnostic visualisation of spatial layers

## Data Inputs

All spatial layers used in this section are stored in the project GeoPackage:

- `gps_tracks_clean` — cleaned GPS movement points  
- `settlements` — settlement boundary polygons  
- `wards` — ward boundary polygons  
- `lgas` — LGA boundary polygons  
- `state` — state boundary polygon  

All layers are reprojected to a common coordinate reference system (CRS) before spatial operations to ensure geometric consistency.

In [42]:
#Libraries needed for this excercise
import pandas as pd
import geopandas as gpd
from pathlib import Path
from shapely.geometry import Point, Polygon
import fiona

In [43]:
# Recreate project paths 
PROJECT_ROOT = Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
gpkg_path = PROCESSED_DIR / "campaign_data.gpkg"

# Load cleaned GPS tracks
tracks_clean = gpd.read_file(gpkg_path, layer="gps_tracks_clean")

# Load reference layers
settlements = gpd.read_file(gpkg_path, layer="settlements")
wards = gpd.read_file(gpkg_path, layer="wards")
lgas = gpd.read_file(gpkg_path, layer="lgas")
state = gpd.read_file(gpkg_path, layer="state")

####  Before any spatial analysis, all datasets must use a consistent Coordinate Reference System (CRS). This validation confirms that the GPS observations and settlement boundaries share compatible spatial reference systems, preventing errors during spatial joins and distance calculations.

In [44]:

print("Tracks CRS:", tracks_clean.crs)
print("Settlements CRS:", settlements.crs)
print("Wards CRS:", wards.crs)
print("LGAs CRS:", lgas.crs)
print("State CRS:", state.crs)


Tracks CRS: EPSG:4326
Settlements CRS: EPSG:4326
Wards CRS: EPSG:4326
LGAs CRS: EPSG:4326
State CRS: EPSG:4326


In [ ]:
#Fix if not aligned
# settlements = settlements.to_crs(tracks_clean.crs)
# wards = wards.to_crs(tracks_clean.crs)
# lgas = lgas.to_crs(tracks_clean.crs)
# state = state.to_crs(tracks_clean.crs)


### Why this step matters
Spatial joins only work when all layers share the same coordinate reference system.
If CRS differs, the join will fail or produce wrong results.

This step ensures:

* settlement polygons line up with GPS points

* ward/LGA/state boundaries align correctly

* attribution is accurate and defensible

In [45]:
## Step 1 — Remove points with latitude/longitude = 0
tracks_clean = tracks_clean[
    (tracks_clean["longitude"] != 0) &
    (tracks_clean["latitude"] != 0)
]


### Step — Validate GPS Coordinates Against the Nigeria Extent
Nigeria roughly spans:

* lon: 2 → 15

* lat: 4 → 14

### GPS devices may occasionally record erroneous coordinates due to signal loss, poor satellite geometry or hardware interruptions. These observations should be identified before spatial attribution..
These errors arise from equipment failures, missing satellite locks, or corrupted fixes.
Such points must be flagged and excluded before spatial attribution, otherwise nearest‑settlement distances become artificially large (hundreds of kilometres).
After filtering out invalid coordinates, nearest‑point attribution produces realistic settlement distances (0–200 m).

In [46]:
tracks_clean = tracks_clean[
    (tracks_clean["longitude"].between(2, 15)) &
    (tracks_clean["latitude"].between(4, 14))
]

### Step 1 — Reproject both layers to EPSG:26391
EPSG:26391 — Minna / Nigeria West Belt
This is a standard Nigerian projected CRS.

In [47]:
tracks_clean_proj = tracks_clean.to_crs("EPSG:3857")
settlements_proj = settlements.to_crs("EPSG:3857")



### Step 2 — Perform nearest spatial join in projected CRS

Reprojecting the data enables distances to be calculated in metres, producing meaningful proximity measurements.

This allows us to do a nearest join


In [48]:
tracks_settlement = gpd.sjoin_nearest(
    tracks_clean_proj,
    settlements_proj,
    how="left",
    distance_col="dist_to_settlement"
)


In [49]:
tracks_settlement["dist_to_settlement"].describe()

count    255635.000000
mean       1577.643269
std        9125.567670
min           0.111319
25%         493.154470
50%        1106.191216
75%        2065.278839
max      519396.264749
Name: dist_to_settlement, dtype: float64

In [50]:
tracks_settlement["invalid_gps"] = tracks_settlement["dist_to_settlement"] > 5000


In [51]:
tracks_settlement["invalid_gps"].sum()


1980

In [52]:
tracks_valid = tracks_settlement[tracks_settlement["dist_to_settlement"] <= 5000]


### Step 3 — A 50-metre tolerance is applied to associate GPS observations with nearby settlements. This recognises that small positioning errors are expected in handheld GPS devices while reducing the likelihood of assigning observations to unrelated settlements.

In [55]:
tracks_valid["visited"] = tracks_valid["dist_to_settlement"] <= 50

C:\Users\Idris\anaconda3\envs\sds2026\Lib\site-packages\geopandas\geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


### The attributed observations are summarised to produce the list of settlements visited during field activities.


In [56]:
visited = (
    tracks_valid[tracks_valid["visited"]]
    .groupby("settlement_id")
    .agg(
        visits=("settlement_id", "count"),
        first_visit=("timestamp", "min"),
        last_visit=("timestamp", "max")
    )
    .reset_index()
)


### All spatial layers were supplied in EPSG:4326 (geographic coordinates in degrees).
Because distance calculations in degrees are not meaningful, both GPS tracks and settlement points were reprojected to EPSG:26391 (Minna / Nigeria West Belt), a projected CRS suitable for Nigeria.
The nearest‑point spatial join and distance tolerance (50 m) were computed in this projected CRS to ensure accurate and defensible results.

In [57]:
visited.shape[0]


1222

In [58]:
visited = (
    tracks_valid[tracks_valid["visited"]]
    .groupby("settlement_id")
    .agg(
        visits=("settlement_id", "count"),
        first_visit=("timestamp", "min"),
        last_visit=("timestamp", "max")
    )
    .reset_index()
)

#### Add settlement names for reporting

In [60]:
visited = visited.merge(
    settlements[["settlement_id", "settlement_name"]],
    on="settlement_id",
    how="left"
)


In [61]:
settlements["visited"] = settlements["settlement_id"].isin(visited["settlement_id"])


In [62]:
settlements["visited"].sum()        # should be 1222
(~settlements["visited"]).sum()     # unvisited


1340

###  Settlement Coverage Summary  
The settlement masterlist contained 2,562 settlements across the campaign LGAs.
GPS tracking identified 1,222 settlements (47.7%) with at least one valid GPS fix within 50 m of the settlement centroid.
A total of 1,340 settlements (52.3%) had no GPS‑verified visit, indicating either:

* the team did not reach the settlement,

* the settlement was inaccessible,

* or GPS logging failed during the visit.

GPS Data Quality  
Out of 255,635 GPS fixes, 1,980 (0.77%) were classified as corrupted due to impossible coordinates or extreme distances (>5 km) from any settlement.
These points were flagged and reported but excluded from settlement‑visit attribution.

## Visit Classification Method  
GPS tracks and settlement points were reprojected to a projected CRS (EPSG:3857) to ensure accurate distance calculations.
A nearest‑point spatial join was used to attribute each GPS fix to the closest settlement.
A tolerance of 50 m was applied to classify a fix as a valid visit, consistent with WHO field guidance and typical GPS drift (5–30 m).